# Python Review: Functions, Classes, Vectorisation and Data Loading/Plotting

**Contents**
1. Functions
2. Classes
3. Vectorisation
4. Creating, saving, loading and plotting data

Cells marked **Example** can simply be run. Cells marked **Exercise** contain `# TODO` gaps for you to complete.

## 0. Setup

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

---
## 1. Functions

### Example 1.1: Mean squared error

$$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

In [ ]:
def mean_squared_error(y_true, y_pred):
    """Return the mean squared error between two sequences of equal length."""
    n = len(y_true)
    total = 0.0
    for i in range(n):
        total += (y_true[i] - y_pred[i]) ** 2
    return total / n

In [ ]:
y_true = [3.0, -0.5, 2.0, 7.0]
y_pred = [2.5,  0.0, 2.0, 8.0]

print("MSE:", mean_squared_error(y_true, y_pred))

### Exercise 1: Mean absolute error

Complete the function below.

$$\text{MAE} = \frac{1}{n}\sum_{i=1}^{n} |y_i - \hat{y}_i|$$

In [ ]:
def mean_absolute_error(y_true, y_pred):
    """Return the mean absolute error between two sequences of equal length."""
    # TODO: compute and return the MAE (hint: abs())
    pass

In [ ]:
# Check your answer (should print True)
print(np.isclose(mean_absolute_error(y_true, y_pred), 0.5))

---
## 2. Classes

### Example 2.0

In [ ]:
# a class is like a template for an object, 
# so Cat here creates a template for cats, 
# and then we can create specific cats from that template, like Tyson and Lily below

class Cat:
    fur = "fluffy" # attributes shared by all instances of the class - all cats have fluffy fur
    
    def __init__(self, name, age): # __init__ always runs when you call a class, and defines the attributes of the object - here, we define the name and the age of the specific cat objects we create (Tyson and Lily)
        self.name = name # self refers to the specific instance of the class, or the "object", so tyson or lily below
        self.age = age

    def food(self, mood="happy"):
        if mood == "happy": # an optional argument with a default value, so if we don't pass in a mood, it will default to "happy"
            if self.age < 5:
                return "I like to eat kitten food."
            else:
                return "I like to eat fish."
        else:
            return "I don't feel like eating right now."

In [ ]:
tyson = Cat("Tyson", 12)

lily = Cat("Lily", 3)

print(tyson.name)
print(lily.age)
print(lily.fur)

### Example 2.1: A simple linear model $\hat{y} = wx + b$

In [ ]:
class LinearModel:
    """A one-dimensional linear model y = w * x + b."""

    def __init__(self, w, b):
        self.w = w
        self.b = b

    def predict(self, x):
        return [self.w * xi + self.b for xi in x]

    def score(self, x, y):
        """Return the MSE of the model's predictions."""
        return mean_squared_error(y, self.predict(x))

In [ ]:
model = LinearModel(w=2.0, b=1.0)

x = [0.0, 1.0, 2.0, 3.0]
y = [1.1, 2.9, 5.2, 6.8]

print("Predictions:", model.predict(x))
print("MSE:        ", model.score(x, y))

### Exercise 2: A standard scaler

Complete the class so that `transform` returns $z = (x - \mu) / \sigma$, where $\mu$ and $\sigma$ are learned in `fit`.

In [ ]:
class StandardScaler:
    """Standardise data to have zero mean and unit standard deviation."""

    def __init__(self):
        self.mean = None
        self.std = None

    def fit(self, x):
        x = np.asarray(x)
        # TODO: store the mean of x in self.mean
        # TODO: store the standard deviation of x in self.std
        return self

    def transform(self, x):
        x = np.asarray(x)
        # TODO: return the standardised values
        pass

In [ ]:
# Check your answer (should print True True)
scaler = StandardScaler().fit([2.0, 4.0, 6.0, 8.0])
z = scaler.transform([2.0, 4.0, 6.0, 8.0])
print(np.isclose(z.mean(), 0.0), np.isclose(z.std(), 1.0))

---
## 3. Vectorisation

### Example 3.1: Timing the dot product

$$\mathbf{a} \cdot \mathbf{b} = \sum_{i=1}^{n} a_i b_i$$

In [ ]:
def dot_loop(a, b):
    total = 0.0
    for i in range(len(a)):
        total += a[i] * b[i]
    return total

In [ ]:
rng = np.random.default_rng(seed=0)
n = 1_000_000
a = rng.standard_normal(n)
b = rng.standard_normal(n)

start = time.perf_counter()
result_loop = dot_loop(a, b)
t_loop = time.perf_counter() - start

start = time.perf_counter()
result_vec = np.dot(a, b)
t_vec = time.perf_counter() - start

print(f"Loop:       {result_loop:.4f}  ({t_loop:.4f} s)")
print(f"Vectorised: {result_vec:.4f}  ({t_vec:.4f} s)")
print(f"Speed-up:   {t_loop / t_vec:.0f}x")

### Exercise 3: Vectorised MSE

Write a vectorised version of `mean_squared_error` (no loops), then time it against the loop version from Section 1.

In [ ]:
def mean_squared_error_vec(y_true, y_pred):
    """Vectorised mean squared error using NumPy."""
    # TODO: compute and return the MSE without a loop
    pass

In [ ]:
y_true = rng.standard_normal(n)
y_pred = rng.standard_normal(n)

# TODO: time mean_squared_error(y_true, y_pred) and store the time in t_loop
t_loop = None

# TODO: time mean_squared_error_vec(y_true, y_pred) and store the time in t_vec
t_vec = None

print(f"Loop:       {t_loop:.4f} s")
print(f"Vectorised: {t_vec:.4f} s")
print("Results match:", np.isclose(mean_squared_error(y_true, y_pred),
                                  mean_squared_error_vec(y_true, y_pred)))

---
## 4. Creating, Saving, Loading and Plotting Data

### Example 4.1: Create a dataset with the following relationship (epsilon is like adding noise)

$$y = 3x_1 - 2x_2 + 0.5x_3 + \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, 0.5^2)$$

In [ ]:
rng = np.random.default_rng(seed=42)
n_samples = 200

x1 = rng.uniform(0, 1, n_samples)
x2 = rng.uniform(0, 1, n_samples)
x3 = rng.normal(0, 1, n_samples)
x4 = rng.normal(0, 1, n_samples)
noise = rng.normal(0, 0.5, n_samples)

y = 3 * x1 - 2 * x2 + 0.5 * x3 + noise

### Example 4.2: Save as CSV

In [ ]:
data = np.column_stack([x1, x2, x3, x4, y])

np.savetxt("data.csv", data, delimiter=",", header="x1,x2,x3,x4,y", comments="")
print("Saved array of shape", data.shape)

### Example 4.3: Load with pandas

In [ ]:
df = pd.read_csv("data.csv")
df.head()

In [ ]:
df.describe()

### Example 4.4: Plot $x_1$ vs $y$

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(df["x1"], df["y"], alpha=0.7)
ax.set_xlabel("$x_1$")
ax.set_ylabel("$y$")
ax.set_title("$x_1$ vs $y$")
plt.show()

### Exercise 4: Explore all features

1. Plot each feature ($x_1, \dots, x_4$) against $y$ on a 2 × 2 grid of subplots.
2. Print the correlation of each feature with $y$.

Which feature appears to have no relationship with $y$?

In [ ]:
features = ["x1", "x2", "x3", "x4"]

fig, axes = plt.subplots(2, 2, figsize=(9, 7))

for ax, feature in zip(axes.flat, features):
    # TODO: scatter plot of df[feature] against df["y"]
    # TODO: add axis labels and a title
    pass

plt.tight_layout()
plt.show()

In [ ]:
# TODO: print the correlation of each feature with y (hint: df.corr())

### Technical - load the data file responses_week1_real.csv and visualise the data 
##### - is the data complete?
##### - what type of data is present? How do you process different types?
##### - which are the most useful data to show? 
##### - plots or tables?
##### - interesting combinations of the data?